# Compute per-site band quality baseline
* Downloads (if not already present) the 30 min low-tide, 0% cloud satellite images per site across 2016-2026
* Computes the per-site, per-band median/MAD baseline (see sentinel2.compute_and_save_band_quality_baseline)
* Only needs to be run once - the baseline is cached to utils.get_band_quality_baseline_path() and reused by sentinel2.flag_anomalous_band_dates

In [1]:
import geopandas
import pathlib
import pandas
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:63925/status,
Dashboard: http://127.0.0.1:63925/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:63927,Workers: 0
Dashboard: http://127.0.0.1:63925/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:63949,Total threads: 2
Dashboard: http://127.0.0.1:63952/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:63930,


# Values to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [4]:
data_path = utils.get_data_path()
utils.create_data_folders()

In [5]:
sites = all_training_sites
mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

# Cells to run
* Download the low-tide no-cloud images by year and site (skips sites already downloaded)
* Compute and save the per-site, per-band quality baseline

### Download satellite images
The 0% cloud, low-tide images across sites for each year and tide delta (30min/45min/1hr),
reusing the prediction-style per-year storage. Pooling across tide deltas (rather than dropping
the low-tide requirement entirely) grows the per-site sample size while keeping the baseline
restricted to genuinely low-tide reflectance.

In [6]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=baseline_max_cloud_cover,
    predict_max_cloud_cover=baseline_max_cloud_cover,
    use_threshold=True, satellite_only=True) 

In [7]:
mndwi_thresholds = pandas.read_csv(mndwi_file)

In [8]:
mndwi_thresholds

,site,threshold
0,Ihutai,0.285990
1,CatlinsLake,0.324971
2,CatlinsRiverMouth,0.077602


In [ ]:
for site in sites:
    mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == site, "threshold",].squeeze()
    if not isinstance(mndwi_threshold, float):
        print(f"WARNING: Expected a float tide threshold for {site_name}. Exiting.")
        break

    # Skip if already run
    if (satellite_path / f"{site}_{years[-1]}_sentinel-2.nc").exists():
        print(f"\tAlready run for {site} as {site}_{years[-1]}_sentinel-2.nc exists. "
              "Delete if you want redownloaded.")
        continue
    
    for year in years:
        print(f"{site}: year {year}")

        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        site_polygon_file = utils.get_site_polygon_path(site)

        if not satellite_file.exists():
            print(f"\tSave satellite image of the site {site} MNDWI thresholded and cloud filtered in the year {year}")
            site_polygon = geopandas.read_file(site_polygon_file)
            satellite_data = sentinel2.get_mndwi_and_cloud_filtered_images_in_year(
                geometry=site_polygon,
                max_cloud_cover=baseline_max_cloud_cover,
                year=year,
                mndwi_threshold=mndwi_threshold,
                apply_anomaly_filter=False,
            )
            if len(satellite_data['time']) == 0:
                print("Warning: No satellite data without cloud. Ignore")
            else:
                print("\t\tSave")
                utils.write_netcdf_conventions_in_place(satellite_data)
                satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                utils.write_netcdf_conventions_in_place(satellite_data)
                utils.save_netcdf(satellite_data, satellite_file)

Ihutai: year 2016
	Save satellite image of the site Ihutai MNDWI thresholded and cloud filtered in the year 2016
	mean MNDWI less than 0.2859899044036865 filtered tiles: 17 from 53 STAC items
	Less than 0% cloud tiles: 0 from the MNDWI filtered tiles of 0. Cloud percentages [ 96.4 100.   99.    0.7  60.5  57.3   1.1  99.1  88.3  56.4  86.2 100.
 100.   99.9   0.2 100.   98.8] Kept [False False False False False False False False False False False False
 False False False False False]
Ihutai: year 2017
	Save satellite image of the site Ihutai MNDWI thresholded and cloud filtered in the year 2017
	mean MNDWI less than 0.2859899044036865 filtered tiles: 23 from 44 STAC items
	Less than 0% cloud tiles: 0 from the MNDWI filtered tiles of 0. Cloud percentages [  0.3   0.5  99.1  85.7  20.4  55.1 100.   89.8  93.3  97.   66.8 100.
  96.6  95.7  89.6 100.  100.   78.3   0.3 100.    3.9 100.    1.3] Kept [False False False False False False False False False False False False
 False False False

### Compute and save the band quality baseline
Pools all downloaded years across all tide deltas per site into one dataset, then computes
per-site, per-band median/MAD and caches it to utils.get_band_quality_baseline_path().

In [ ]:
baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=baseline_sites,
    satellite_images_paths=list(satellite_paths.values()),
)
baseline

In [ ]:
import pandas
pandas.read_csv(r"C:\Local\repos\seagrass-detection\data\band_quality_baseline_old.csv")